# Fingrid – Save and Load XML

Build a `ReserveBid_MarketDocument` for Fingrid, write it to
`examples/data/fingrid_reserve_bid.xml`, then read it back and
deserialize it with `deserialize_reserve_bid_document()`.

In [ ]:
from pathlib import Path

from nexa_mfrr_eam import (
    Bid,
    BidDocument,
    BiddingZone,
    MarketProductType,
    MARIMode,
    TSO,
    deserialize_reserve_bid_document,
)

# --- build ---
bid = (
    Bid.up(volume_mw=40, price_eur=68.50)
    .divisible(min_volume_mw=10)
    .for_mtu("2026-03-21T10:00Z")
    .resource("10XHYDRO-FI-001-A", coding_scheme="A01")
    .bidding_zone(BiddingZone.FI)
    .product_type(MarketProductType.SCHEDULED_AND_DIRECT)
    .build()
)

doc = (
    BidDocument(tso=TSO.FINGRID)
    .sender(party_id="10XBSP-FI-001---A", coding_scheme="A01")
    .add_bid(bid)
    .build()
)

assert not doc.validate(mari_mode=MARIMode.PRE_MARI), "validation errors"

# --- save ---
out_path = Path("data/fingrid_reserve_bid.xml")
out_path.write_bytes(doc.to_xml())
print(f"Saved {out_path.stat().st_size} bytes → {out_path}")

In [ ]:
# --- load and deserialize ---
parsed = deserialize_reserve_bid_document(out_path.read_bytes())

bts = parsed.bid_time_series[0]
print(f"Document mRID: {parsed.mrid}")
print(f"Receiver EIC:  {parsed.receiver_mrid}")
print(f"MTU:           {bts.period.time_interval_start.strftime('%Y-%m-%dT%H:%MZ')}")
print(f"Volume:        {bts.period.point.quantity} MW")
print(f"Price:         {bts.period.point.energy_price} EUR/MWh")
print(f"Round-trip ok: {parsed.mrid == doc.model.mrid}")